# NB17 — S5 public-HF completion audit and report (CPU)

Run after NB14, NB15 and NB16 finish. This notebook does not retrain models.
It checks all81 jobs against a pinned HF snapshot, checkpoint hashes, complete
60-epoch histories, held-out prediction coverage, and recomputed paired ROI F1.
Partial results are reported as partial, never green. The per-image localisation
and frozen-classifier evaluation is performed by the training notebooks in a
separate process after each trained model, so a failed evaluation needs no retraining.

SAM2/manual comparison remains deferred. S9 is not completed by S5.


In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# Writes tyrelib.py into the session and imports it. Nothing here touches the
# GPU or the network beyond installing three small packages.
#
#   tyrelib   the whole pipeline: HuggingFace sync, registry, work sharding,
#             telemetry, model zoo, training loop, metrics.
#
# Generated by build_notebooks.py from tyrelib.py. Editing the blob below does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle ships torch, pandas, sklearn. These vary by image version, so check.
#   pynvml  reads GPU power/temperature/clocks directly (per device)
#   psutil  peak RAM and CPU
#   pyarrow writes per-sample predictions as Parquet
for _pkg in ('pynvml', 'psutil', 'pyarrow'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg],
                       check=False)

_LIB = (
    '...',
)

(WORK / 'tyrelib.py').write_bytes(base64.b64decode(''.join(_LIB)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
# Without this, re-running cell 1 after an edit returns the cached module and
# you spend an hour debugging a ghost.
for _m in [m for m in list(sys.modules) if m == 'tyrelib']:
    del sys.modules[_m]
import tyrelib as tl
print('tyrelib', tl.__version__, 'loaded')


In [ ]:
(WORK/'s5_data.py').write_bytes(base64.b64decode('IiIiUzUgbWFudWFsLWxhYmVsIHByb3RvY29sIGFuZCBnZW9tZXRyeS4gTm8gR1BVLCB0cmFpbmluZywgb3IgbmV0d29yayBzaWRlIGVmZmVjdHMuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgpSRVZJU0lPTiA9ICdzNS1tYW51YWwtMjAyNi0wOS0xMC1yMScKU09VUkNFX1JFVklTSU9OID0gJ2RkNDNiMjMxY2ZiZGQ5MmRkNmQ4YzAxYjQ3MTY2ZGRlYzRhYjA1ZjgnCk1PREVMUyA9IHsKICAgICd1bmV0X3IzNCc6ICgnc2VtYW50aWMnLCAnc21wLlVuZXQ6cmVzbmV0MzQ6aW1hZ2VuZXQnKSwKICAgICdkZWVwbGFidjNwbHVzX3IzNCc6ICgnc2VtYW50aWMnLCAnc21wLkRlZXBMYWJWM1BsdXM6cmVzbmV0MzQ6aW1hZ2VuZXQnKSwKICAgICdzZWdmb3JtZXJfYjAnOiAoJ3NlbWFudGljJywgJ252aWRpYS9taXQtYjAnKSwKICAgICdzZWdmb3JtZXJfYjInOiAoJ3NlbWFudGljJywgJ252aWRpYS9taXQtYjInKSwKICAgICd5b2xvMjZuX2RldCc6ICgneW9sbycsICd5b2xvMjZuLnB0JyksCiAgICAneW9sbzI2c19kZXQnOiAoJ3lvbG8nLCAneW9sbzI2cy5wdCcpLAogICAgJ3lvbG8yNm5fc2VnJzogKCd5b2xvJywgJ3lvbG8yNm4tc2VnLnB0JyksCiAgICAneW9sbzI2c19zZWcnOiAoJ3lvbG8nLCAneW9sbzI2cy1zZWcucHQnKSwKICAgICdydGRldHJ2Ml9yMTgnOiAoJ3J0ZGV0cicsICdQZWtpbmdVL3J0ZGV0cl92Ml9yMTh2ZCcpLAp9ClBBQ0tBR0VTID0geyd1bHRyYWx5dGljcyc6ICc4LjQuMjAnLCAndHJhbnNmb3JtZXJzJzogJzQuNTEuMycsCiAgICAgICAgICAgICdzZWdtZW50YXRpb24tbW9kZWxzLXB5dG9yY2gnOiAnMC41LjAnLCAndGltbSc6ICcxLjAuMTUnLAogICAgICAgICAgICAncHljb2NvdG9vbHMnOiAnMi4wLjExJ30KCgpkZWYgc2lnbmF0dXJlKHZhbHVlKToKICAgIHJldHVybiBoYXNobGliLnNoYTI1Nihqc29uLmR1bXBzKHZhbHVlLCBzb3J0X2tleXM9VHJ1ZSwgc2VwYXJhdG9ycz0oJywnLCAnOicpKS5lbmNvZGUoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgZGlnZXN0KHBhdGgpOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggb3BlbihwYXRoLCAncmInKSBhcyBmOgogICAgICAgIGZvciBibG9jayBpbiBpdGVyKGxhbWJkYTogZi5yZWFkKDEwMjQgKiAxMDI0KSwgYicnKToKICAgICAgICAgICAgaC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gaC5oZXhkaWdlc3QoKQoKCmRlZiByZWdpb25zKG1hc2spOgogICAgaWYgbWFzay5uZGltICE9IDIgb3Igbm90IG5wLmlzaW4obWFzaywgWzAsIDEsIDIsIDMsIDRdKS5hbGwoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdFeHBlY3RlZCBhbiBpbmRleGVkIG1hbnVhbCBtYXNrIHdpdGggdmFsdWVzIDAuLjQnKQogICAgcmV0dXJuIG5wLnN0YWNrKChtYXNrID4gMCwgKG1hc2sgPT0gMikgfCAobWFzayA9PSAzKSkpCgoKZGVmIGJveChtYXNrKToKICAgIHksIHggPSBucC53aGVyZShtYXNrKQogICAgcmV0dXJuIFtpbnQoeC5taW4oKSksIGludCh5Lm1pbigpKSwgaW50KHgubWF4KCkpICsgMSwgaW50KHkubWF4KCkpICsgMV0gaWYgbGVuKHgpIGVsc2UgTm9uZQoKCmRlZiBwYWRkZWRfYm94KGIsIHdpZHRoLCBoZWlnaHQsIGZyYWN0aW9uPS4wNSk6CiAgICBpZiBiIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIFswLCAwLCB3aWR0aCwgaGVpZ2h0XQogICAgeDAsIHkwLCB4MSwgeTEgPSBtYXAoZmxvYXQsIGIpCiAgICBpZiBub3QgbnAuaXNmaW5pdGUoYikuYWxsKCkgb3IgeDEgPD0geDAgb3IgeTEgPD0geTA6CiAgICAgICAgcmV0dXJuIFswLCAwLCB3aWR0aCwgaGVpZ2h0XQogICAgcHgsIHB5ID0gZnJhY3Rpb24gKiAoeDEteDApLCBmcmFjdGlvbiAqICh5MS15MCkKICAgIHJlc3VsdCA9IFttYXgoMCwgaW50KG5wLmZsb29yKHgwLXB4KSkpLCBtYXgoMCwgaW50KG5wLmZsb29yKHkwLXB5KSkpLAogICAgICAgICAgICAgIG1pbih3aWR0aCwgaW50KG5wLmNlaWwoeDErcHgpKSksIG1pbihoZWlnaHQsIGludChucC5jZWlsKHkxK3B5KSkpXQogICAgcmV0dXJuIHJlc3VsdCBpZiByZXN1bHRbMl0gPiByZXN1bHRbMF0gYW5kIHJlc3VsdFszXSA+IHJlc3VsdFsxXSBlbHNlIFswLCAwLCB3aWR0aCwgaGVpZ2h0XQoKCmRlZiBtYXNrX3Njb3JlcyhyZWZlcmVuY2UsIHByZWRpY3Rpb24pOgogICAgaW1wb3J0IGN2MgogICAgYSwgYiA9IG5wLmFzYXJyYXkocmVmZXJlbmNlLCBib29sKSwgbnAuYXNhcnJheShwcmVkaWN0aW9uLCBib29sKQogICAgaWYgYS5zaGFwZSAhPSBiLnNoYXBlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ05hdGl2ZSBtYXNrIGdlb21ldHJ5IGRpZmZlcnMnKQogICAgaW50ZXIsIHVuaW9uLCB0b3RhbCA9IGludCgoYSAmIGIpLnN1bSgpKSwgaW50KChhIHwgYikuc3VtKCkpLCBpbnQoYS5zdW0oKStiLnN1bSgpKQogICAga2VybmVsID0gbnAub25lcygoMywgMyksIG5wLnVpbnQ4KQogICAgZWEgPSBhICYgfmN2Mi5lcm9kZShhLmFzdHlwZSgndWludDgnKSwga2VybmVsLCBib3JkZXJUeXBlPWN2Mi5CT1JERVJfQ09OU1RBTlQsIGJvcmRlclZhbHVlPTApLmFzdHlwZShib29sKQogICAgZWIgPSBiICYgfmN2Mi5lcm9kZShiLmFzdHlwZSgndWludDgnKSwga2VybmVsLCBib3JkZXJUeXBlPWN2Mi5CT1JERVJfQ09OU1RBTlQsIGJvcmRlclZhbHVlPTApLmFzdHlwZShib29sKQogICAgbmVhcl9hID0gY3YyLmRpbGF0ZShlYS5hc3R5cGUoJ3VpbnQ4JyksIG5wLm9uZXMoKDUsIDUpLCBucC51aW50OCkpLmFzdHlwZShib29sKQogICAgbmVhcl9iID0gY3YyLmRpbGF0ZShlYi5hc3R5cGUoJ3VpbnQ4JyksIG5wLm9uZXMoKDUsIDUpLCBucC51aW50OCkpLmFzdHlwZShib29sKQogICAgcHJlY2lzaW9uID0gZmxvYXQoKGViICYgbmVhcl9hKS5zdW0oKS9tYXgoMSwgZWIuc3VtKCkpKQogICAgcmVjYWxsID0gZmxvYXQoKGVhICYgbmVhcl9iKS5zdW0oKS9tYXgoMSwgZWEuc3VtKCkpKQogICAgYmYgPSAyKnByZWNpc2lvbipyZWNhbGwvKHByZWNpc2lvbityZWNhbGwpIGlmIHByZWNpc2lvbityZWNhbGwgZWxzZSAwLgogICAgcmV0dXJuIGRpY3QoaW91PWludGVyL3VuaW9uIGlmIHVuaW9uIGVsc2UgTm9uZSwgZGljZT0yKmludGVyL3RvdGFsIGlmIHRvdGFsIGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgIGJvdW5kYXJ5X2YxXzJweD1iZiBpZiB0b3RhbCBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICByZWZlcmVuY2VfcGl4ZWxzPWludChhLnN1bSgpKSwgcHJlZGljdGlvbl9waXhlbHM9aW50KGIuc3VtKCkpKQoKCmRlZiBpbnNwZWN0X2RhdGEocm9vdCwgYW5ub3RhdGlvbnMpOgogICAgcm9vdCwgYW5ub3RhdGlvbnMgPSBQYXRoKHJvb3QpLCBQYXRoKGFubm90YXRpb25zKQogICAgY2xlYW4gPSBwZC5yZWFkX2Nzdihyb290LydtYW5pZmVzdHMvY2xlYW5fbWFuaWZlc3QuY3N2JykKICAgIGFzc2VydCBsZW4oY2xlYW4pID09IDQxOCBhbmQgY2xlYW4uaW1hZ2VfaWQuaXNfdW5pcXVlLCAnRXhwZWN0ZWQgNDE4IHVuaXF1ZSBjbGVhbiBpbWFnZXMnCiAgICBhc3NlcnQgc2V0KGNsZWFuLmltYWdlX2tpbmQpID09IHsnY2xlYW5fb3JpZ2luYWwnfQogICAgcmVjb3JkcyA9IFtdCiAgICBmb3Igcm93IGluIGNsZWFuLnNvcnRfdmFsdWVzKCdpbWFnZV9pZCcpLml0ZXJ0dXBsZXMoKToKICAgICAgICBpbWFnZV9wYXRoLCBtYXNrX3BhdGggPSByb290L3Jvdy5yZWxhdGl2ZV9wYXRoLCBhbm5vdGF0aW9ucy8nY2xlYW4vbWFza3MnL2Yne3Jvdy5pbWFnZV9pZH0ucG5nJwogICAgICAgIHdpdGggSW1hZ2Uub3BlbihpbWFnZV9wYXRoKSBhcyBpbSwgSW1hZ2Uub3BlbihtYXNrX3BhdGgpIGFzIG1tOgogICAgICAgICAgICBhc3NlcnQgaW0uc2l6ZSA9PSBtbS5zaXplLCBmJ0ltYWdlL21hc2sgc2l6ZSBtaXNtYXRjaDoge3Jvdy5pbWFnZV9pZH0nCiAgICAgICAgICAgIHdpZHRoLCBoZWlnaHQgPSBpbS5zaXplCiAgICAgICAgICAgIHJyID0gcmVnaW9ucyhucC5hcnJheShtbSkpCiAgICAgICAgYXNzZXJ0IHJyWzBdLmFueSgpIGFuZCByclsxXS5hbnkoKSwgZidNaXNzaW5nIHR5cmUvdHJlYWQ6IHtyb3cuaW1hZ2VfaWR9JwogICAgICAgIGltYWdlX3NoYSA9IGRpZ2VzdChpbWFnZV9wYXRoKQogICAgICAgIGFzc2VydCBpbWFnZV9zaGEgPT0gcm93LmZpbGVfc2hhMjU2LCBmJ0ltYWdlIGNvbnRlbnRzIGRpZmZlciBmcm9tIG1hbmlmZXN0OiB7cm93LmltYWdlX2lkfScKICAgICAgICByZWNvcmRzLmFwcGVuZChkaWN0KGltYWdlX2lkPXJvdy5pbWFnZV9pZCwgaW1hZ2Vfc2hhMjU2PWltYWdlX3NoYSwgbWFza19zaGEyNTY9ZGlnZXN0KG1hc2tfcGF0aCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB3aWR0aD13aWR0aCwgaGVpZ2h0PWhlaWdodCwgdHlyZV9ib3g9Ym94KHJyWzBdKSwgdHJlYWRfYm94PWJveChyclsxXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBwcm94eV9sYWJlbD1yb3cucHJveHlfbGFiZWwsIHNlc3Npb25fZ3JvdXA9cm93LnNlc3Npb25fZ3JvdXAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICByZWxhdGl2ZV9wYXRoPXJvdy5yZWxhdGl2ZV9wYXRoLCBmb2xkX2lkPWludChyb3cuZm9sZF9pZCkpKQogICAgc3BsaXRzID0ge30KICAgIGtub3duID0gc2V0KGNsZWFuLmltYWdlX2lkKQogICAgZm9yIGZvbGQgaW4gKDAsIDEsIDIpOgogICAgICAgIHRyID0gcGQucmVhZF9jc3Yocm9vdC9mJ3NwbGl0cy9jdntmb2xkfV90cmFpbi5jc3YnKQogICAgICAgIHZhID0gcGQucmVhZF9jc3Yocm9vdC9mJ3NwbGl0cy9jdntmb2xkfV92YWxpZGF0aW9uLmNzdicpCiAgICAgICAgdHIgPSB0ci5sb2NbdHIuaW1hZ2Vfa2luZC5lcSgnY2xlYW5fb3JpZ2luYWwnKV0KICAgICAgICBhc3NlcnQgc2V0KHZhLmltYWdlX2tpbmQpID09IHsnY2xlYW5fb3JpZ2luYWwnfQogICAgICAgIGFzc2VydCB0ci5pbWFnZV9pZC5pc191bmlxdWUgYW5kIHZhLmltYWdlX2lkLmlzX3VuaXF1ZQogICAgICAgIGFzc2VydCBzZXQodHIuaW1hZ2VfaWQpLmlzZGlzam9pbnQodmEuaW1hZ2VfaWQpCiAgICAgICAgYXNzZXJ0IHNldCh0ci5zZXNzaW9uX2dyb3VwKS5pc2Rpc2pvaW50KHZhLnNlc3Npb25fZ3JvdXApCiAgICAgICAgYXNzZXJ0IHNldCh0ci5pbWFnZV9pZCkgfCBzZXQodmEuaW1hZ2VfaWQpID09IGtub3duCiAgICAgICAgYXNzZXJ0IHNldCh0ci5maWxlX3NoYTI1NikuaXNkaXNqb2ludCh2YS5maWxlX3NoYTI1NiksICdFeGFjdCBpbWFnZSBsZWFrYWdlJwogICAgICAgIHNwbGl0c1tzdHIoZm9sZCldID0gZGljdCh0cmFpbj1zb3J0ZWQodHIuaW1hZ2VfaWQpLCB2YWxpZGF0aW9uPXNvcnRlZCh2YS5pbWFnZV9pZCkpCiAgICByZXR1cm4gZGljdChyZWNvcmRzPXJlY29yZHMsIHNwbGl0cz1zcGxpdHMsCiAgICAgICAgICAgICAgICBtYW5pZmVzdF9zaGEyNTY9ZGlnZXN0KHJvb3QvJ21hbmlmZXN0cy9jbGVhbl9tYW5pZmVzdC5jc3YnKSwKICAgICAgICAgICAgICAgIHNwbGl0X3NoYTI1Nj17Zidjdntmb2xkfV97cm9sZX0nOmRpZ2VzdChyb290L2Ync3BsaXRzL2N2e2ZvbGR9X3tyb2xlfS5jc3YnKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgZm9sZCBpbiAoMCwxLDIpIGZvciByb2xlIGluICgndHJhaW4nLCd2YWxpZGF0aW9uJyl9KQoKCmRlZiBwcm90b2NvbChkYXRhKToKICAgIHJldHVybiBkaWN0KHJldmlzaW9uPVJFVklTSU9OLCBzb3VyY2VfcmV2aXNpb249U09VUkNFX1JFVklTSU9OLAogICAgICAgIGltcGxlbWVudGF0aW9uX3NoYTI1Nj17bmFtZTpkaWdlc3QoUGF0aChfX2ZpbGVfXykucGFyZW50L25hbWUpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgbmFtZSBpbiAoJ3M1X2RhdGEucHknLCdzNV9ydW50aW1lLnB5Jyl9LAogICAgICAgIG1vZGVscz17azogbGlzdCh2KSBmb3IgaywgdiBpbiBNT0RFTFMuaXRlbXMoKX0sIHBhY2thZ2VzPVBBQ0tBR0VTLAogICAgICAgIGVwb2Nocz02MCwgZm9sZHM9WzAsIDEsIDJdLCBzZWVkcz1bMSwgMiwgM10sIGxhYmVsX3NvdXJjZT0nZXhpc3RpbmdfbWFudWFsJywKICAgICAgICBkYXRhPWRhdGEsIHRyYWluaW5nX2ltYWdlcz0nY2xlYW5fb25seV9ub19kZXJpdmVkX2ltYWdlcycsCiAgICAgICAgcmVnaW9ucz1bJ3R5cmU6IGxhYmVscyAxKzIrMys0JywgJ3RyZWFkOiBsYWJlbHMgMiszJ10sCiAgICAgICAgc2VtYW50aWM9ZGljdChzaXplPTUxMiwgYmF0Y2g9NCwgbG9zcz0nQkNFV2l0aExvZ2l0cyArIHNvZnQgRGljZSAodHdvIG92ZXJsYXBwaW5nIHNpZ21vaWQgY2hhbm5lbHMpJywKICAgICAgICAgICAgICAgICAgICAgIG9wdGltaXplcj0nQWRhbVcnLCBscj0uMDAwMSwgd2VpZ2h0X2RlY2F5PS4wMSwgc2NoZWR1bGVyPSdjb3NpbmUnLCBhdWdtZW50YXRpb249J2hvcml6b250YWxfZmxpcF8wLjUnKSwKICAgICAgICBydGRldHI9ZGljdChzaXplPTUxMiwgYmF0Y2g9Miwgb3B0aW1pemVyPSdBZGFtVycsIGxyPS4wMDAxLCB3ZWlnaHRfZGVjYXk9LjAxLAogICAgICAgICAgICAgICAgICAgIHNjaGVkdWxlcj0nY29zaW5lJywgYXVnbWVudGF0aW9uPSdub25lJywgbG9zcz0nbmF0aXZlIFJULURFVFJ2MiBkZXRlY3Rpb24gbG9zcycpLAogICAgICAgIHlvbG89ZGljdChzaXplPTUxMiwgYmF0Y2g9NCwgb3B0aW1pemVyPSdBZGFtVycsIGxyPS4wMDAxLCB3ZWlnaHRfZGVjYXk9LjAxLAogICAgICAgICAgICAgICAgICBhdWdtZW50YXRpb249J2hvcml6b250YWxfZmxpcF8wLjVfb25seScsIHBvbHlnb25fbWluX2lvdT0uOTgsCiAgICAgICAgICAgICAgICAgIG92ZXJsYXBfbWFzaz1GYWxzZSwgZW5kcG9pbnQ9J2ZpbmFsX2Vwb2NoX0VNQScpLAogICAgICAgIGVuZHBvaW50PSdmaXhlZF9lcG9jaF82MF9ub3RfdmFsaWRhdGlvbl9zZWxlY3RlZCcsIGdwdT0nY3VkYTowX25vX0RhdGFQYXJhbGxlbCcsCiAgICAgICAgZG93bnN0cmVhbT1kaWN0KGNsYXNzaWZpZXI9J2EtcmVzbmV0NTAtYmFzZS1me2ZvbGR9LXN7c2VlZH0nLCBjaGVja3BvaW50PSdja3B0X2xhc3QucHQnLAogICAgICAgICAgICAgICAgICAgICAgICBjbGFzc2lmaWVyX3NvdXJjZV9yZXZpc2lvbj1TT1VSQ0VfUkVWSVNJT04sIGNyb3BfcGFkZGluZz0uMDUsCiAgICAgICAgICAgICAgICAgICAgICAgIG1vZGVzPVsnZnVsbCcsICdwcmVkX3R5cmUnLCAncHJlZF90cmVhZCcsICdvcmFjbGVfdHlyZScsICdvcmFjbGVfdHJlYWQnXSwKICAgICAgICAgICAgICAgICAgICAgICAgbWlzc2luZ19wcmVkaWN0aW9uPSdmdWxsX2ltYWdlX2ZhbGxiYWNrJywgY2xhc3NpZmllcl90cmFpbmluZz0nbm9uZV9mcm96ZW4nKSwKICAgICAgICBsaW1pdGF0aW9ucz1bJ2ZvbGRzIDAvMiBzdXNwZWN0ZWQgY3Jvc3MtdHlyZSBsZWFrYWdlOyBmb2xkIDEgdGlueSB0eXJlIHNhbXBsZScsCiAgICAgICAgICAgICAgICAgICAgICAnZGVzY3JpcHRpdmUgZXhpc3RpbmctZm9sZCBldmFsdWF0aW9uLCBub3QgaW5kZXBlbmRlbnQgbmV3LXR5cmUgdGVzdGluZycsCiAgICAgICAgICAgICAgICAgICAgICAnU0FNMiBjb21wYXJpc29uIGFuZCBibGluZCByZXBlYXQgYW5ub3RhdGlvbiBkZWZlcnJlZCcsCiAgICAgICAgICAgICAgICAgICAgICAnYmFja2VuZC1uYXRpdmUgbG9zc2VzL0VNQS9wcmV0cmFpbmluZyBkaWZmZXI7IG5vdCBhIGNvbnRyb2xsZWQgZXF1YWwtcHJldHJhaW5pbmcgYWJsYXRpb24nXSkKCgpkZWYgam9icyhwbGFuLCBmYW1pbHk9Tm9uZSk6CiAgICByZXN1bHQgPSBbXQogICAgZm9yIG5hbWUsIChiYWNrZW5kLCBfKSBpbiBwbGFuWydtb2RlbHMnXS5pdGVtcygpOgogICAgICAgIGlmIGZhbWlseSBhbmQgYmFja2VuZCAhPSBmYW1pbHk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm9yIGZvbGQgaW4gcGxhblsnZm9sZHMnXToKICAgICAgICAgICAgZm9yIHNlZWQgaW4gcGxhblsnc2VlZHMnXToKICAgICAgICAgICAgICAgIHJlc3VsdC5hcHBlbmQoZGljdChtb2RlbD1uYW1lLCBiYWNrZW5kPWJhY2tlbmQsIGZvbGQ9Zm9sZCwgc2VlZD1zZWVkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJ1bl9pZD1mJ3tuYW1lfS1me2ZvbGR9LXN7c2VlZH0nKSkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgYXNzaWduZWQocGxhbiwgZmFtaWx5LCB3b3JrZXIsIHdvcmtlcnMpOgogICAgaWYgbm90IDAgPD0gd29ya2VyIDwgd29ya2VyczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdXb3JrZXIgb3V0c2lkZSBhY3RpdmUgYWNjb3VudCBsaXN0JykKICAgICMgT3duZXJzaGlwIG5ldmVyIGRlcGVuZHMgb24gd2hpY2ggam9icyBoYXZlIGFscmVhZHkgZmluaXNoZWQuCiAgICByZXR1cm4gW2ogZm9yIGksIGogaW4gZW51bWVyYXRlKGpvYnMocGxhbiwgZmFtaWx5KSkgaWYgaSAlIHdvcmtlcnMgPT0gd29ya2VyXQoKCmRlZiBzcGxpdF9mcmFtZXMocGxhbiwgcm9vdCwgZm9sZCk6CiAgICBjbGVhbiA9IHBkLnJlYWRfY3N2KFBhdGgocm9vdCkvJ21hbmlmZXN0cy9jbGVhbl9tYW5pZmVzdC5jc3YnKS5zZXRfaW5kZXgoJ2ltYWdlX2lkJywgZHJvcD1GYWxzZSkKICAgIHMgPSBwbGFuWydkYXRhJ11bJ3NwbGl0cyddW3N0cihmb2xkKV0KICAgIHJldHVybiBjbGVhbi5sb2Nbc1sndHJhaW4nXV0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKSwgY2xlYW4ubG9jW3NbJ3ZhbGlkYXRpb24nXV0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQo='))
(WORK/'s5_runtime.py').write_bytes(base64.b64decode('...'))
(WORK/'s5_notebook.py').write_bytes(base64.b64decode('...'))
import importlib
import s5_data as sd
import s5_notebook as sn
importlib.reload(sd); importlib.reload(sn)


## Session — Internet ON, HF_TOKEN secret; one notebook by default

In [ ]:
# === Who am I? =============================================================
#
# ACCOUNT labels this Kaggle account in the shared run log.
# ACTIVE_KAGGLE_ACCOUNTS is the ONE source of truth for parallelism.
# NUM_WORKERS and WORKER_ID are derived from it; do not edit them.
#
# ---------------------------------------------------------------------------
# THESE TWO VALUES ARE SAFE TO CHANGE AT ANY TIME.
#
# They decide which account owns each fresh run. An absent or partial run stays
# with that static owner. Work stealing is disabled by default; an explicit
# recovery run may opt in and can then take over only a real claim/run event
# older than 45 minutes. Whether a run is finished, and what epoch it reached,
# is read from HuggingFace -- from the run's own files -- so it is the same
# answer for every account at every worker count.
# Go from 4 workers to 1 and nothing is retrained: the runs the other three
# finished are skipped, and the ones they left half-done are RESUMED from
# their checkpoints.
#
# (It did not always work that way. Resume used to check only the local disk,
#  and Kaggle wipes that between sessions, so every run restarted at epoch 1.
#  See docs/05 -- Bug 8.)
# ---------------------------------------------------------------------------
#
# All accounts push to the SAME HuggingFace account (Shanmuk4622), so the
# 128-writes-per-hour budget is SHARED. tyrelib caps each worker at
# 100/NUM_WORKERS automatically.
#
# DEFAULT: exactly one Kaggle notebook. For four parallel copies, replace the
# tuple with ('acct1', 'acct2', 'acct3', 'acct4') in every copy and set ACCOUNT
# to that copy's label. Keeping the active labels in one tuple prevents a cell
# that says "one worker" in one place but silently launches as worker 0/4.
ACTIVE_KAGGLE_ACCOUNTS = ('acct1',)   # <<< one notebook; list all four only when all four run
ACCOUNT = 'acct1'                     # <<< this copy's label

# A missing comma in ('acct1') makes it a string. tyrelib deliberately repairs
# that common edit, so a valid one-worker session cannot fail before HF sync.
ACTIVE_KAGGLE_ACCOUNTS = tl.normalise_active_accounts(ACTIVE_KAGGLE_ACCOUNTS)
if ACCOUNT not in ACTIVE_KAGGLE_ACCOUNTS:
    raise ValueError(f"ACCOUNT={ACCOUNT!r} is not active: {ACTIVE_KAGGLE_ACCOUNTS}")
NUM_WORKERS = len(ACTIVE_KAGGLE_ACCOUNTS)
WORKER_ID = ACTIVE_KAGGLE_ACCOUNTS.index(ACCOUNT)
print(f"RUN MODE CHECK: {ACCOUNT=} {ACTIVE_KAGGLE_ACCOUNTS=} -> worker {WORKER_ID}/{NUM_WORKERS}")

sess = tl.Session(account=ACCOUNT, worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                  stage='s5',
                  hf_repo='Shanmuk4622/tyre-wear-study',
                  enable_hf=True,
                  session_limit_h=8.5,      # push + pause before Kaggle kills us
                  push_interval_min=30)     # background commit cycle


In [ ]:
# === Find the dataset ======================================================
# One Kaggle dataset holds the whole package:
#     <slug>/FINAL/{images,splits,manifests}
#     <slug>/annotations/{clean,propagated}
# Kaggle sometimes wraps uploads in one more directory, so both are searched for.
DATA_ROOT = sess.prepare_data()
ANN_ROOT  = tl.find_annotations_root(DATA_ROOT)
print("annotations:", ANN_ROOT if ANN_ROOT else "NOT FOUND (only needed from NB08 onward)")


In [ ]:
assert ANN_ROOT is not None, 'Attach Tire Dataset Prepared including annotations/clean/masks'


In [ ]:
PREFIX = ''  # automatically find the unique matching NB13 protocol on HF
PLAN = sn.load_plan(sess, PREFIX, DATA_ROOT, ANN_ROOT)
PREFIX = sn.plan_prefix(PLAN)


In [ ]:
REPORT = sn.report(sess, PLAN, PREFIX)
assert sess.finish()
print(REPORT.to_string(index=False))